In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("D:\\projects\\SWIIGGY ZOMATO\\Rider-Info.csv")

In [2]:
df.head()
df.shape
df.columns
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 450000 entries, 0 to 449999
Data columns (total 20 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_time            450000 non-null  object 
 1   order_id              450000 non-null  int64  
 2   order_date            450000 non-null  object 
 3   allot_time            450000 non-null  object 
 4   accept_time           449843 non-null  object 
 5   pickup_time           447579 non-null  object 
 6   delivered_time        444782 non-null  object 
 7   rider_id              450000 non-null  int64  
 8   first_mile_distance   450000 non-null  float64
 9   last_mile_distance    450000 non-null  float64
 10  alloted_orders        433052 non-null  float64
 11  delivered_orders      432659 non-null  float64
 12  cancelled             450000 non-null  int64  
 13  undelivered_orders    432659 non-null  float64
 14  lifetime_order_count  449947 non-null  float64
 15  

In [3]:
df.isnull().sum()

order_time                   0
order_id                     0
order_date                   0
allot_time                   0
accept_time                157
pickup_time               2421
delivered_time            5218
rider_id                     0
first_mile_distance          0
last_mile_distance           0
alloted_orders           16948
delivered_orders         17341
cancelled                    0
undelivered_orders       17341
lifetime_order_count        53
reassignment_method     436256
reassignment_reason     436247
reassigned_order        436247
session_time              3675
cancelled_time          444782
dtype: int64

In [4]:
df.drop(columns=[
    'reassignment_method',
    'reassignment_reason',
    'reassigned_order',
    'cancelled_time'
], inplace=True)

In [5]:
df = df.dropna(subset=['pickup_time', 'delivered_time'])

In [6]:
df['accept_time'].fillna(method='ffill', inplace=True)

C:\Users\princ\AppData\Local\Temp\ipykernel_7880\1818915906.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['accept_time'].fillna(method='ffill', inplace=True)
C:\Users\princ\AppData\Local\Temp\ipykernel_7880\1818915906.py:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['accept_time'].fillna(method='ffill', inplace=True)


In [7]:
df.isnull().sum()

order_time                  0
order_id                    0
order_date                  0
allot_time                  0
accept_time                 0
pickup_time                 0
delivered_time              0
rider_id                    0
first_mile_distance         0
last_mile_distance          0
alloted_orders          16299
delivered_orders        16486
cancelled                   0
undelivered_orders      16486
lifetime_order_count        2
session_time             3531
dtype: int64

In [8]:
df.drop(columns=[
    'alloted_orders',
    'delivered_orders',
    'undelivered_orders'
], inplace=True)

In [9]:
df['session_time'].fillna(df['session_time'].median(), inplace=True)

C:\Users\princ\AppData\Local\Temp\ipykernel_7880\3233837634.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['session_time'].fillna(df['session_time'].median(), inplace=True)


In [10]:
df['lifetime_order_count'].fillna(df['lifetime_order_count'].median(), inplace=True)

C:\Users\princ\AppData\Local\Temp\ipykernel_7880\3254975674.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['lifetime_order_count'].fillna(df['lifetime_order_count'].median(), inplace=True)


In [11]:
df.isnull().sum()

order_time              0
order_id                0
order_date              0
allot_time              0
accept_time             0
pickup_time             0
delivered_time          0
rider_id                0
first_mile_distance     0
last_mile_distance      0
cancelled               0
lifetime_order_count    0
session_time            0
dtype: int64

In [12]:
df['total_distance'] = df['first_mile_distance'] + df['last_mile_distance']

In [14]:
df['order_time'] = pd.to_datetime(df['order_time'])
df['delivered_time'] = pd.to_datetime(df['delivered_time'])

df['delivery_time'] = (df['delivered_time'] - df['order_time']).dt.total_seconds() / 60

In [15]:
df['hour'] = df['order_time'].dt.hour

In [16]:
def time_slot(x):
    if x < 12:
        return "Morning"
    elif x < 18:
        return "Afternoon"
    else:
        return "Evening"

df['time_slot'] = df['hour'].apply(time_slot)

In [17]:
df['is_peak'] = df['hour'].apply(lambda x: 1 if 18 <= x <= 22 else 0)

In [18]:
def distance_bucket(x):
    if x < 2:
        return "Short"
    elif x < 5:
        return "Medium"
    else:
        return "Long"

df['distance_category'] = df['total_distance'].apply(distance_bucket)

In [19]:
def risk(row):
    score = 0
    if row['total_distance'] > 5:
        score += 2
    if row['is_peak'] == 1:
        score += 3
    return score

df['risk_score'] = df.apply(risk, axis=1)

In [20]:
df.to_csv("cleaned_orders.csv", index=False)